# Reproducing the TinyStories experiments (Appendix S)

This notebook runs the same pipeline as the command-line scripts in the repository root:

- **Part A** rebuilds every table, number and the figure from the released logs in `results/` (CPU, under a minute).
- **Part B** re-runs all 37 training / evaluation jobs from scratch (GPU, ≈ 17.6 T4 GPU-hours; ≈ 9–10 h on 2× T4)
  and checks the new numbers against the paper.

Jobs are resumable: if the session stops, re-run the same cell and finished jobs are skipped while unfinished ones
resume from their last 1,000-step checkpoint.

In [ ]:
import os, sys, subprocess
# Run from the repository root (the directory containing lh.py).
if not os.path.exists("lh.py") and os.path.exists("../lh.py"):
    os.chdir("..")
assert os.path.exists("lh.py"), "open this notebook from the repository (lh.py not found)"
print("working directory:", os.getcwd())

def sh(cmd):
    # Run a command and stream its output into the notebook.
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout: print(line, end="")
    if p.wait() != 0: raise RuntimeError(f"command failed: {cmd}")

## Part A: tables and figure from the released logs

In [ ]:
sh("python analyze.py --out results --check")

In [ ]:
import pandas as pd
from IPython.display import Image, display
pd.set_option("display.width", 200)
display(pd.read_csv("results/analysis/screen.csv").round(3))
display(pd.read_csv("results/analysis/steering_per_seed.csv")[
    ["ctx", "d_h", "seed", "ppl_cost", "r_never", "r_late", "h", "h_copy", "h_hard", "posthoc_vs_late", "deletable"]].round(3))
display(Image("results/analysis/tinystories_handover.png"))

## Part B: full re-run (GPU)

Install the requirements if needed (`pip install -r requirements.txt`), then run the stages in order.
Each stage depends on the previous one; `--stage all` runs everything in one call.

In [ ]:
OUT = "runs"          # training outputs: data, checkpoints, logs
import torch
print("torch", torch.__version__, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
sh(f"python run_experiments.py --list --out {OUT}")

### Data preparation (CPU, ≈ 10 min): tokenise TinyStories and check it against the paper corpus

In [ ]:
sh(f"python run_experiments.py --stage prepare --out {OUT}")

### Screen: free hybrid and recurrent-only model at four (context, GRU width) conditions, seed 0 (≈ 7.7 GPU-h)

In [ ]:
sh(f"python run_experiments.py --stage screen --out {OUT}")

### Steering: never and late (λ = 0.1) arms from every seed-0 checkpoint, plus λ = 0.3 at 64/512 (≈ 2.9 GPU-h)

In [ ]:
sh(f"python run_experiments.py --stage steer --out {OUT}")

### Seeds 1–2 at the lowest- and highest-viability conditions (≈ 6.9 GPU-h)

In [ ]:
sh(f"python run_experiments.py --stage seeds --out {OUT}")

### Matched post-hoc attenuation control (minutes)

In [ ]:
sh(f"python run_experiments.py --stage posthoc --out {OUT}")

### Analysis of the re-run, compared with the paper

In [ ]:
sh(f"python analyze.py --out {OUT} --check")
display(Image(f"{OUT}/analysis/tinystories_handover.png"))

### Monitoring a long run

Each job writes its evaluation log to `runs/logs/<run>.jsonl` (one line per 1,000 steps) and its console output to
`runs/stdout/`. The cell below prints the latest line of every log; it can be run from a second session or after
reconnecting.

In [ ]:
import json, glob
for f in sorted(glob.glob(f"{OUT}/logs/*.jsonl")):
    lines = [json.loads(l) for l in open(f) if l.strip()]
    if lines:
        r = lines[-1]
        print(f"{os.path.basename(f)[:-6]:24s} step {r['step']:6d}  ppl {r['ppl']:.3f}  copy {r['copy_acc']:.3f}  "
              f"deletion ratio {r.get('del_ppl_ratio', float('nan')):.3f}")